In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

ROOT = "/content/drive/MyDrive/Dissertation/Dataset"
DATA_PATH = Path(ROOT) / "data" / "features" / "video_windowed" / "DEAP_video_windowed_features_thr5p5.npz"

z = np.load(DATA_PATH, allow_pickle=True)
X = z["X"].astype(np.float32)
y = z["y"].astype(int)
groups = z["groups"].astype(object)

print("Loaded:", DATA_PATH)
print("X:", X.shape, "y:", y.shape)
print("Class balance:", np.unique(y, return_counts=True))
print("Unique subjects:", len(np.unique(groups)))


Mounted at /content/drive
Loaded: /content/drive/MyDrive/Dissertation/Dataset/data/features/video_windowed/DEAP_video_windowed_features_thr5p5.npz
X: (1280, 284) y: (1280,)
Class balance: (array([0, 1]), array([731, 549]))
Unique subjects: 32


In [2]:
SEED = 42

def build_models():
    models = {}

    # Logistic Regression
    models["LogReg"] = Pipeline([
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            solver="lbfgs",
            max_iter=4000,
            random_state=SEED
        ))
    ])

    # Linear SVM
    models["LinearSVM"] = Pipeline([
        ("sc", StandardScaler()),
        ("clf", LinearSVC(
            class_weight="balanced",
            C=1.0,
            random_state=SEED
        ))
    ])

    # RBF SVM
    models["SVM_RBF"] = Pipeline([
        ("sc", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,   # needed for mean-prob aggregation if you later do window-level
            random_state=SEED
        ))
    ])

    # Random Forest
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED
    )

    # Gradient Boosting
    models["GradBoost"] = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED
    )

    # XGBoost (if installed)
    if HAVE_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            eval_metric="logloss",
        )

    return models

models = build_models()
print("Models:", list(models.keys()))


Models: ['LogReg', 'LinearSVM', 'SVM_RBF', 'RandomForest', 'GradBoost', 'XGBoost']


In [3]:
def groupkfold_eval(X, y, groups, models, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)

    summary_rows = []
    all_fold_details = []

    for name, model in models.items():
        accs, f1s = [], []
        cms = []

        for fold, (tr, te) in enumerate(gkf.split(X, y, groups), start=1):
            clf = model
            clf.fit(X[tr], y[tr])
            pred = clf.predict(X[te])

            acc = accuracy_score(y[te], pred)
            f1m = f1_score(y[te], pred, average="macro")
            cm = confusion_matrix(y[te], pred)

            accs.append(acc)
            f1s.append(f1m)
            cms.append(cm)

            all_fold_details.append({
                "model": name,
                "fold": fold,
                "n_train": len(tr),
                "n_test": len(te),
                "acc": acc,
                "macro_f1": f1m,
                "cm": cm
            })

        summary_rows.append({
            "model": name,
            "acc_mean": float(np.mean(accs)),
            "acc_std":  float(np.std(accs)),
            "macro_f1_mean": float(np.mean(f1s)),
            "macro_f1_std":  float(np.std(f1s)),
            "n_splits": n_splits
        })

    summary = pd.DataFrame(summary_rows).sort_values("macro_f1_mean", ascending=False)
    details = pd.DataFrame(all_fold_details)
    return summary, details

summary, details = groupkfold_eval(X, y, groups, models, n_splits=5)

display(summary.assign(
    acc_mean=lambda d: d.acc_mean.round(3),
    acc_std=lambda d: d.acc_std.round(3),
    macro_f1_mean=lambda d: d.macro_f1_mean.round(3),
    macro_f1_std=lambda d: d.macro_f1_std.round(3),
))


,model,acc_mean,acc_std,macro_f1_mean,macro_f1_std,n_splits
1,LinearSVM,0.540,0.027,0.531,0.028,5
0,LogReg,0.531,0.017,0.521,0.024,5
4,GradBoost,0.533,0.036,0.500,0.041,5
5,XGBoost,0.512,0.031,0.485,0.024,5
2,SVM_RBF,0.485,0.046,0.477,0.046,5
3,RandomForest,0.560,0.029,0.427,0.031,5


In [4]:
best_model = summary.iloc[0]["model"]
print("BEST model:", best_model)

best_rows = details[details["model"] == best_model].sort_values("fold")
cms = best_rows["cm"].to_list()

cm_sum = np.sum(cms, axis=0)
print("Aggregated confusion matrix across folds:\n", cm_sum)

print("\nFold-by-fold:")
display(best_rows[["fold","n_train","n_test","acc","macro_f1"]].assign(
    acc=lambda d: d.acc.round(3),
    macro_f1=lambda d: d.macro_f1.round(3)
))


BEST model: LinearSVM
Aggregated confusion matrix across folds:
 [[383 348]
 [241 308]]

Fold-by-fold:


,fold,n_train,n_test,acc,macro_f1
5,1,1000,280,0.546,0.546
6,2,1000,280,0.539,0.529
7,3,1040,240,0.529,0.519
8,4,1040,240,0.583,0.571
9,5,1040,240,0.500,0.488
